In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import duckdb
import numpy as np
from mplsoccer import Pitch, VerticalPitch
from plottable import ColumnDefinition, Table
from plottable.cmap import normed_cmap
from matplotlib.colors import PowerNorm
from matplotlib.lines import Line2D
from matplotlib.lines import Line2D
from mplsoccer import VerticalPitch
import matplotlib.patches as mpatches
import math

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

from great_tables import GT

from extract_carries import extract_carries
import xg_model_baked

from scipy.stats import zscore
from scipy.stats import norm


*Create metrics class*
        
        *Subtask: simple xG model*
        *Subtask: extrapolate carries*

Create Z-Score Creation Class

Create Export Table Class

Forwards
    npXg
    xG assisted (cba figuring out xA fully, will never have to do that in the real world)
    *shots*
    *npGoals*
    *Aerial %*
    *EPV passing*
    *Dribble/Takeon rate*
    *Carrying*
    *Touches in final third*
    Through balls received
    *Through balls played*
    *Fouls won*

Midfielders
    *Progressive pass distance*
    *Progressive passes*
    *Progressive carries*
    *Passes into penalty area*
    *Total tackles*
    *Successful tackles*
    *Tackle %*
    *Pass completion %*
    *Forward passes*
    *Forward pass %*
    *Interceptions (tackles and interceptions?)*
    *Touches*


Defenders
    *Blocked shots*
    *Recoveries*
    *Blocked Crosses*
    *Clearances*

In [35]:
class Metrics:
    def __init__(self, file_path: str, conditions: str = None, players_tablename: str = 'players'):
        """
        file_path is path to DuckDB database. conditions is a string for the
        WHERE clause if a specific range of events is needed.
        """
        con = duckdb.connect(file_path)
        if conditions is None:
            self.events = con.sql("select * from events").df()
        else:
            self.events = con.sql(f"select * from events where {conditions}").df()
        self.events['playerId'] = self.events['playerId'].fillna(-1).astype('int64')  # keep start/end slides for through-ball math, give them an int id
        self.players = con.sql(f"select * from {players_tablename}").df()
        con.close()

        self.events = xg_model_baked.add_xg(self.events)

    def create_metrics(self):
        ev = self.events
        is_shot = ev["type"].isin(["Goal", "SavedShot", "MissedShots", "ShotOnPost"])
        pens = ev[["penaltyScored", "penaltyMissed"]].fillna(False).astype(bool).any(axis=1)

        shots = ev[is_shot]
        shooting_agg = (
            shots.assign(np_shot=(~pens[is_shot]).astype(int),
                         np_goal=((shots["type"] == "Goal") & ~pens[is_shot]).astype(int))
            .groupby("playerId")
            .agg(
                shots=("np_shot", "sum"),
                np_goals=("np_goal", "sum"),
                npxg=("xG", "sum"),
                big_chances_missed=("bigChanceMissed", "sum"),
                big_chances_scored=("bigChanceScored", "sum"),
            )
        )
        shooting_agg["big_chances"] = (
            shooting_agg["big_chances_missed"] + shooting_agg["big_chances_scored"]
        )

        dribbling_agg = (
            ev.groupby("playerId")
            .agg(
                dribbles_lost=("dribbleLost", "sum"),
                dribbles_won=("dribbleWon", "sum"),
            )
        )
        dribbling_agg["dribbles_total"] = (
            dribbling_agg["dribbles_lost"] + dribbling_agg["dribbles_won"]
        )
        dribbling_agg["dribble_success"] = (
            dribbling_agg["dribbles_won"] / dribbling_agg["dribbles_total"].replace(0, np.nan)
        ) * 100

        passing_df = ev[ev["type"] == "Pass"].copy()
        keypass_cols = ["keyPassLong", "keyPassShort", "keyPassCross",
                        "keyPassThroughball", "keyPassOther"]
        passing_df["isKeyPassOP"] = passing_df[keypass_cols].fillna(False).any(axis=1)
        passing_df["isCompleted"] = passing_df["outcomeType"] == "Successful"

        passing_agg_basic = (
            passing_df.groupby("playerId")
            .agg(
                key_passes_open_play=("isKeyPassOP", "sum"),
                through_balls_completed=("passThroughBallAccurate", "sum"),
                passes=("isTouch", "sum"),
                completed_passes=("isCompleted", "sum"),
                pass_epv=("EPV", "sum"),
                forward_passes=("passForward", "sum"),
                big_chances_created=("bigChanceCreated", "sum"),
            )
        )
        passing_agg_basic["completion_percentage"] = (
            passing_agg_basic["completed_passes"]
            / passing_agg_basic["passes"].replace(0, np.nan)
        ) * 100
        passing_agg_basic["forward_pct"] = (
            passing_agg_basic["forward_passes"]
            / passing_agg_basic["passes"].replace(0, np.nan)
        ) * 100

        ev_sorted = ev.sort_values(["matchId", "minute", "second", "id"])
        ev_sorted["prev_type"] = ev_sorted.groupby("matchId")["type"].shift(1)
        for c in ["keyPassCorner", "keyPassFreekick", "keyPassThrowin"]:
            ev_sorted[f"prev_{c}"] = ev_sorted.groupby("matchId")[c].shift(1)

        shot_rows = ev_sorted[ev_sorted["assist_playerId"].notna()].copy()
        shot_rows["assist_set_piece"] = shot_rows[
            ["prev_keyPassCorner", "prev_keyPassFreekick", "prev_keyPassThrowin"]
        ].fillna(False).astype(bool).any(axis=1)

        DEF_ACTIONS = ["Tackle", "Interception", "BallRecovery", "Clearance", "Challenge"]
        def_ft = ev[(ev["type"].isin(DEF_ACTIONS)) & (ev["x"] >= 66.666667)]
        def_ft_agg = (
            def_ft.groupby("playerId")
            .size()
            .rename("def_actions_final_third")
            .to_frame()
        )

        xga_agg = (
            shot_rows.groupby("assist_playerId")["xG_assisted"]
            .sum().rename("xg_assisted")
        )
        xga_op_agg = (
            shot_rows[~shot_rows["assist_set_piece"]]
            .groupby("assist_playerId")["xG_assisted"]
            .sum().rename("xg_assisted_op")
        )
        for s in (xga_agg, xga_op_agg):
            s.index = s.index.astype("int64")
            s.index.name = "playerId"

        adv = ev[(ev["type"] == "Pass") & (ev["outcomeType"] == "Successful")].copy()
        adv["dist_start"] = np.sqrt((100 - adv["x"]) ** 2 + (50 - adv["y"]) ** 2)
        adv["dist_end"] = np.sqrt((100 - adv["endX"]) ** 2 + (50 - adv["endY"]) ** 2)
        adv["progressive_passing_distance"] = (adv["endX"] - adv["x"]).clip(lower=0.0)
        adv["isProgressivePass"] = (adv["x"] > 33.333333) & (
            adv["dist_start"] <= 0.75 * adv["dist_end"]
        )
        adv["penAreaEnd"] = (adv["endX"] >= 83) & (adv["endY"] >= 21) & (adv["endY"] <= 79)

        passing_agg_advanced = (
            adv.groupby("playerId")
            .agg(
                progressive_passing_distance=("progressive_passing_distance", "sum"),
                progressive_passes=("isProgressivePass", "sum"),
                passes_into_pen_area=("penAreaEnd", "sum"),
            )
        )

        defense_agg = (
            ev.groupby("playerId")
            .agg(
                interceptions=("interceptionWon", "sum"),
                tackles_won=("tackleWon", "sum"),
                tackles_lost=("tackleLost", "sum"),
                errors_leading_to_goal=("errorLeadsToGoal", "sum"),
                errors_leading_to_shot=("errorLeadsToShot", "sum"),
                clearances=("clearanceEffective", "sum"),
                recoveries=("ballRecovery", "sum"),
                blocks=("outfielderBlock", "sum"),
                blocked_crosses=("passCrossBlockedDefensive", "sum"),
                aerial_wins=("duelAerialWon", "sum"),
                aerial_losses=("duelAerialLost", "sum"),
                interceptions_in_box=("interceptionIntheBox", "sum"),
            )
        )
        defense_agg["tackles_total"] = defense_agg["tackles_won"] + defense_agg["tackles_lost"]
        defense_agg["tackle_success"] = (
            defense_agg["tackles_won"] / defense_agg["tackles_total"].replace(0, np.nan)
        ) * 100
        defense_agg["aerials"] = defense_agg["aerial_wins"] + defense_agg["aerial_losses"]
        defense_agg["aerial_success"] = (
            defense_agg["aerial_wins"] / defense_agg["aerials"].replace(0, np.nan)
        ) * 100

        DEF_ACTIONS = ["Tackle", "Interception", "BallRecovery", "Clearance", "BlockedPass"]
        def_ft = ev[(ev["type"].isin(DEF_ACTIONS)) & (ev["x"] >= 66.666667)]
        def_ft_agg = (
            def_ft.groupby("playerId")
            .size()
            .rename("def_actions_final_third")
            .to_frame()
        )
        
        touches = ev[ev["isTouch"] == True]
        touches_agg = (
            touches.groupby("playerId")
            .agg(
                touches=("playerId", "count"),
                touches_in_final_third=("finalThird", "sum"),
                fouls_won=("foulGiven", "sum"),
            )
        )

        carries = extract_carries(ev, 2.0)
        carries["progressive_carrying_distance"] = (
            carries["endX"] - carries["startX"]
        ).clip(lower=0.0)
        carries["isProgressive"] = (
            ((carries["startX"] < 60) & (carries["endX"] >= 10.5 + carries["startX"]))
            | ((carries["startX"] > 60) & (carries["endX"] >= 5.25 + carries["startX"]))
            | ((carries["endX"] >= 83) & (carries["endY"] >= 21) & (carries["endY"] <= 79))
        )
        carries["intoPenArea"] = (
            (carries["endX"] >= 83) & (carries["endY"] >= 21) & (carries["endY"] <= 79)
        )
        carries_agg = (
            carries.groupby("playerId")
            .agg(
                progressive_carrying_distance=("progressive_carrying_distance", "sum"),
                progressive_carries=("isProgressive", "sum"),
                carries_into_penalty_area=("intoPenArea", "sum"),
            )
        )

        player_info = self.players[['playerId', 'playerName', 'teamName', 'minutes', 'position']]
        player_info = player_info.set_index('playerId')

        metrics = pd.concat(
            [player_info, shooting_agg, dribbling_agg, passing_agg_basic,
             passing_agg_advanced, defense_agg, touches_agg, carries_agg,
             xga_agg, xga_op_agg, def_ft_agg],
            axis=1,
        )


        metrics = metrics.drop(index=[-1, 0], errors='ignore').fillna(0.0)

        return metrics
    
    def generate_normalized_metrics(self):
        """
        normalizes applicable metrics to per 90 (figuring out TIP/Otip soon)
        """
        df = self.create_metrics().copy()

        
        df['minutes'] = df['minutes'].astype(str).str.replace(',', '').astype(float)

        exclude = {
        'playerName', 'teamName', 'minutes',
        'completion_percentage', 'forward_pct', 'dribble_success',
        'tackle_success', 'aerial_success',
            }
        cols = [c for c in df.columns
            if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]

        nineties = df['minutes'] / 90
        df[cols] = df[cols].div(nineties, axis=0)
        
        return df

    def generate_ratings(self, df_data, position: str, metric_names_and_weights: dict, inclusive: bool = False, minutes_cutoff: int = 200):
        """
        Data input should be normalized

        Metric names and weights should be structured as a dictionary in the below format
        {npxg:30, shots:30}...

        Possible positions: 'FW','MF','DF','GK'
        """

        total = sum(metric_names_and_weights.values())
        if total != 100:
            raise ValueError(f'values must add up to 100. Your values add to {total}')

        demographic_cols = ['playerName', 'teamName', 'position', 'minutes']
        stat_cols = list(metric_names_and_weights.keys())

        cols = demographic_cols + stat_cols

        df = df_data.copy()

        if inclusive == False:
            df = df[df['position'].fillna('').str.contains(position, na=False)]
        else:
            df = df[df['position'] == position]

        df = df[df['minutes'] >= minutes_cutoff]

        df = df[cols]
        df_zscores = df.copy()

        for col in stat_cols:
            df_zscores[f'{col}_zscore'] = zscore(df_zscores[col], nan_policy='omit')

        weights = pd.Series(metric_names_and_weights)

        zscore_cols = [f'{metric}_zscore' for metric in weights.index]

        df_zscores['rating'] = (
            df_zscores[zscore_cols]
            .mul(weights.values, axis=1)
            .sum(axis=1)
            / weights.sum()
        )

        df_zscores['rating_100'] = (
            norm.cdf(df_zscores['rating']) * 100
        )

        return df_zscores

In [37]:
obj = Metrics('/Users/owner/Desktop/Natabase/mls.duckdb', conditions= 'YEAR(startDate) = 2026', players_tablename= 'players_2026_fbref')
df = obj.generate_normalized_metrics()




/Users/owner/Desktop/GitHub/NextStepsFootballAnallytics/CompRatingArticle/MLSCompRatingArticle/xg_model_baked.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prev = ev.groupby("matchId")["keyPassThroughball"].shift(1).fillna(False).astype(int)
/var/folders/hq/8f5kvgjd5dx23mthw16_z1lc0000gn/T/ipykernel_23726/2031859572.py:89: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ].fillna(False).astype(bool).any(axis=1)


In [38]:
df1 = obj.generate_ratings(position='MF', df_data= df, inclusive = False, minutes_cutoff= 400, metric_names_and_weights={'npxg': 5,'xg_assisted':10, 'tackles_won': 5, 'tackle_success': 5, 'interceptions': 5, 'progressive_passing_distance': 15, 'progressive_passes' :15, 'progressive_carrying_distance': 5, 'passes_into_pen_area': 10, 'aerial_success': 5, 'pass_epv': 15, 'touches': 5}).sort_values('rating_100', ascending = False)

In [39]:
df1

,playerName,teamName,position,minutes,npxg,xg_assisted,tackles_won,tackle_success,interceptions,progressive_passing_distance,...,interceptions_zscore,progressive_passing_distance_zscore,progressive_passes_zscore,progressive_carrying_distance_zscore,passes_into_pen_area_zscore,aerial_success_zscore,pass_epv_zscore,touches_zscore,rating,rating_100
playerId,,,,,,,,,,,,,,,,,,,,,
393132.0,Sebastian Berhalter,Vancouver W'caps,MF,1121.0,0.241115,0.281266,1.445138,51.428571,0.883140,352.621766,...,0.035286,1.904495,1.084019,0.579788,2.498191,1.096685,3.283046,2.031968,1.685319,95.403644
573792.0,Pedro Soma,San Diego FC,MF,400.0,0.039615,0.113258,0.900000,66.666667,0.675000,494.640000,...,-0.355734,3.407606,3.183000,8.743407,0.042026,-1.270969,1.068174,3.571569,1.681940,95.370976
551861.0,Jeppe Tverskov,San Diego FC,MF,845.0,0.073466,0.118091,1.810651,73.913043,2.023669,457.157396,...,2.177931,3.010893,0.954190,5.697638,0.142764,0.332669,0.882496,4.360199,1.487580,93.156910
303909.0,Joaquín Pereyra,Minnesota Utd,"FW,MF",1277.0,0.171414,0.185919,0.704777,71.428571,1.268598,275.588880,...,0.759424,1.089185,1.355756,-0.180977,2.372668,-0.484124,3.088467,1.088065,1.270529,89.805186
11119.0,Lionel Messi,Inter Miami,"FW,MF",1242.0,0.707579,0.365302,0.362319,55.555556,0.144928,255.181159,...,-1.351550,0.873191,-0.212117,0.943458,3.354256,-0.039789,1.701156,0.668124,1.229721,89.059915
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
318923.0,Derrick Etienne,Toronto FC,"MF,FW",646.0,0.181316,0.038548,0.696594,71.428571,0.557276,66.023220,...,-0.576896,-1.128843,-0.648661,-0.837649,-0.906719,-1.490822,-1.431495,-1.656916,-0.852680,19.691842
344069.0,Sergio Córdova,St. Louis City,"FW,MF",495.0,0.135500,0.009696,0.909091,45.454545,0.181818,49.436364,...,-1.282246,-1.304397,-0.720561,-0.595427,-0.724008,0.332669,-1.073735,-1.481455,-0.858457,19.531998
406255.0,Gabriel Pirani,D.C. United,"FW,MF",468.0,0.075169,0.025452,0.192308,25.000000,0.192308,53.461538,...,-1.262540,-1.261795,1.426968,-0.648422,-0.844264,-2.150383,-1.547939,-1.342867,-0.860731,19.469299


In [40]:
df2_wideattacker = obj.generate_ratings(position='FW', df_data= df, inclusive = False, minutes_cutoff= 400, metric_names_and_weights={'npxg': 15,'xg_assisted_op':35, 'dribble_success': 10, 'dribbles_total': 20, 'shots': 5, 'pass_epv': 5, 'through_balls_completed': 5, 'progressive_carrying_distance': 5}).sort_values('rating_100', ascending = False)

In [41]:
df2_wideattacker

,playerName,teamName,position,minutes,npxg,xg_assisted_op,dribble_success,dribbles_total,shots,pass_epv,...,npxg_zscore,xg_assisted_op_zscore,dribble_success_zscore,dribbles_total_zscore,shots_zscore,pass_epv_zscore,through_balls_completed_zscore,progressive_carrying_distance_zscore,rating,rating_100
playerId,,,,,,,,,,,,,,,,,,,,,
11119.0,Lionel Messi,Inter Miami,"FW,MF",1242.0,0.707579,0.350995,47.222222,5.217391,6.014493,0.163891,...,2.196633,4.139964,0.500831,2.276132,4.138432,1.702989,2.734683,1.280001,2.776597,99.725344
323490.0,Evander,FC Cincinnati,"MF,FW",1109.0,0.501265,0.222374,46.428571,4.544635,3.814247,0.165368,...,1.037560,2.047619,0.446082,1.733309,1.637358,1.726113,3.785892,-0.032498,1.619414,94.732089
393093.0,Cade Cowell,RB New York,"FW,MF",879.0,0.233696,0.302547,28.125000,3.276451,2.662116,0.117246,...,-0.465648,3.351832,-0.816564,0.710057,0.327702,0.972505,-0.694111,0.484141,1.218161,88.841858
22221.0,Luis Suárez,Inter Miami,FW,614.0,0.621585,0.283559,22.222222,1.319218,3.371336,0.099909,...,1.713519,3.042943,-1.223759,-0.869163,1.133889,0.701003,1.617812,0.321267,1.214548,88.773069
120342.0,Tyler Boyd,LAFC,"MF,FW",448.0,0.162842,0.233251,50.000000,4.017857,2.611607,0.054321,...,-0.863703,2.224556,0.692452,1.308271,0.270287,-0.012909,0.890175,-0.700818,1.002276,84.189473
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
344069.0,Sergio Córdova,St. Louis City,"FW,MF",495.0,0.135500,0.009696,40.000000,1.818182,1.272727,0.005673,...,-1.017313,-1.412155,0.002616,-0.466567,-1.251650,-0.774762,-0.694111,-0.720101,-0.911934,18.090164
445287.0,Nathan Ordaz,LAFC,"FW,MF",502.0,0.283083,0.005991,33.333333,1.075697,1.434263,-0.016978,...,-0.188187,-1.472415,-0.457275,-1.065652,-1.068029,-1.129481,-0.694111,-1.029258,-0.998475,15.902451
406255.0,Gabriel Pirani,D.C. United,"FW,MF",468.0,0.075169,0.025452,7.692308,2.500000,0.961538,-0.021365,...,-1.356253,-1.155838,-2.226084,0.083567,-1.605386,-1.198187,-0.694111,-0.788979,-1.028210,15.192563


In [42]:
df3_striker = obj.generate_ratings(position='FW', df_data= df, inclusive = False, minutes_cutoff= 400, metric_names_and_weights={'npxg': 30,'xg_assisted_op':5, 'shots': 20, 'np_goals': 20, 'pass_epv': 5, 'aerial_success': 10, 'def_actions_final_third':10 }).sort_values('rating_100', ascending = False)

In [43]:
df3_striker

,playerName,teamName,position,minutes,npxg,xg_assisted_op,shots,np_goals,pass_epv,aerial_success,def_actions_final_third,npxg_zscore,xg_assisted_op_zscore,shots_zscore,np_goals_zscore,pass_epv_zscore,aerial_success_zscore,def_actions_final_third_zscore,rating,rating_100
playerId,,,,,,,,,,,,,,,,,,,,
11119.0,Lionel Messi,Inter Miami,"FW,MF",1242.0,0.707579,0.350995,6.014493,0.797101,0.163891,40.000000,1.449275,2.196633,4.139964,4.138432,1.734064,1.702989,0.134280,0.057789,2.144844,98.401732
296870.0,Sam Surridge,Nashville SC,FW,457.0,0.978498,0.011735,3.544858,1.575492,-0.003171,37.500000,0.984683,3.718661,-1.378980,1.331136,4.709256,-0.913253,-0.002120,-0.710957,2.137758,98.373179
404198.0,Petar Musa,FC Dallas,FW,1040.0,0.762289,0.067597,4.153846,0.865385,0.014045,56.818182,2.076923,2.503997,-0.470245,2.023389,1.995058,-0.643647,1.051876,1.096336,1.714015,95.673702
315387.0,Hugo Cuypers,Chicago Fire,FW,989.0,0.859993,0.049835,3.276036,1.001011,-0.002948,36.363636,2.002022,3.052899,-0.759182,1.025561,2.513454,-0.909772,-0.064119,0.972400,1.631053,94.856044
361474.0,Brian White,Vancouver W'caps,FW,1197.0,0.841977,0.065689,3.909774,0.676692,-0.008113,54.000000,1.127820,2.951688,-0.501280,1.745946,1.273830,-0.990647,0.898116,-0.474113,1.457265,92.747844
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
344069.0,Sergio Córdova,St. Louis City,"FW,MF",495.0,0.135500,0.009696,1.272727,0.181818,0.005673,47.058824,0.909091,-1.017313,-1.412155,-1.251650,-0.617692,-0.774762,0.519407,-0.836036,-0.820071,20.608777
512617.0,Joaquín Valiente,FC Dallas,"MF,FW",863.0,0.096247,0.136478,1.147161,0.000000,0.090594,18.181818,1.251448,-1.237836,0.650288,-1.394385,-1.312644,0.555138,-1.056115,-0.269549,-0.985052,16.229938
295950.0,Manu García,Sporting KC,"MF,FW",1002.0,0.027569,0.166305,0.538922,0.089820,0.148626,0.000000,1.437126,-1.623673,1.135503,-2.085785,-0.969330,1.463926,-2.048111,0.037685,-1.169196,12.116237
